# Tome 4, Chapter 1: Finite Calculus (Python companion)

Exercises first, solutions at the end. We let SymPy find closed forms and check those of the chapter, compute Bernoulli numbers exactly and redo Bernoulli's famous sum, recover the cost of loops from a few measurements with Newton's formula, build a difference engine, simulate cycles of permutations against the Stirling numbers, and test the theorem of von Staudt and Clausen far beyond what one can do by hand.

*Before each solution, predict what the output will look like.*

## Exercises

**P1.1 (★) Closed forms by computer.** Ask SymPy for $\sum_{k=0}^nk^22^k$, $\sum_{k=0}^nkH_k$ and $\sum_{k=1}^n\frac1{k(k+1)(k+2)}$, and compare with Exercises 1.1, 1.3 and 1.2. When SymPy fails, check the chapter's formula exactly for $n\le30$.

**P1.2 (★) Bernoulli's half quarter of an hour.** Compute $B_0,\dots,B_{30}$ exactly from the recurrence of Proposition 1.14(a), then $\sum_{k=1}^{1000}k^{10}$ with Faulhaber's formula (Theorem 1.15) and directly. Bernoulli printed this number in 1713: can you read off its structure from the formula?

**P1.3 (★★) CS application: the cost of a loop, from measurements.** Instrument a triple loop `for i<n, for j<i, for l<j` and insertion sort on a reversed list, count the basic operations for $n=0,\dots,7$, and recover the exact cost as a polynomial with Newton's formula (Theorem 1.16). Check the prediction at $n=100$. What could go wrong if the cost were not a polynomial?

**P1.4 (★★) A difference engine.** Write Babbage's machine: registers holding $f(0),\Delta f(0),\dots,\Delta^df(0)$, each step adding every register to the one above it. Tabulate Euler's polynomial $f(t)=t^2+t+41$ for $t<50$ with additions only, and count how many of the values $f(0),\dots,f(39)$ are prime.

**P1.5 (★★) Stirling numbers and random permutations.** Fill the tables of Stirling numbers with the recurrences of Proposition 1.11, check the explicit formula for $\stirlingii nk$, and compare the distribution of the number of cycles of $200\,000$ random permutations of $10$ elements with $\stirlingi{10}k/10!$.

**P1.6 (★★★) Von Staudt–Clausen and irregular primes.** Check Exercise 1.7 for all even $m\le200$. Then list the *irregular* primes below $200$: those $p\ge5$ dividing the numerator of some $B_k$ with $k$ even, $2\le k\le p-3$. Kummer proved in 1850 that Fermat's last theorem holds for every exponent that is a regular prime.

## Solutions

### P1.1

In [ ]:
import sympy as sp
k, n = sp.symbols('k n', integer=True, nonnegative=True)
for expr in [k**2 * 2**k, k * sp.harmonic(k), 1 / (k * (k + 1) * (k + 2))]:
    lo = 1 if expr.has(1 / k) else 0
    s = sp.simplify(sp.summation(expr, (k, lo, n)))
    print(expr, '  ->  ', s)
# compare with the closed forms of the chapter for n = 0..30
assert all(sum(j*j*2**j for j in range(N+1)) == (N*N-2*N+3)*2**(N+1)-6 for N in range(31))
assert all(sum(j*sp.harmonic(j) for j in range(N)) == sp.Rational(N*(N-1), 2)*(sp.harmonic(N)-sp.Rational(1, 2)) for N in range(1, 31))
print('closed forms of Exercises 1.1 and 1.3 checked for n <= 30')

SymPy finds the first and third sums, in forms that expand to $(n^2-2n+3)2^{n+1}-6$ and $\frac14-\frac1{2(n+1)(n+2)}=\frac{n(n+3)}{4(n+1)(n+2)}$. It returns the second unevaluated: summation by parts with harmonic numbers is not in its toolbox, and the closed form of Exercise 1.3 is checked exactly instead. Computer algebra systems sum such expressions with algorithms (Gosper's, then Zeilberger's) that decide whether a *hypergeometric* closed form exists; $kH_k$ is not hypergeometric, so a human with summation by parts wins here.

### P1.2

In [ ]:
def bernoulli_minus(m):            # convention B_1 = -1/2 of the chapter
    B = [sp.Rational(1)]
    for N in range(1, m + 1):
        B.append(-sum(sp.binomial(N + 1, j) * B[j] for j in range(N)) / (N + 1))
    return B
B = bernoulli_minus(30)
print([str(b) for b in B[:13]])
def faulhaber(p, N):
    return sum(sp.binomial(p + 1, j) * B[j] * N**(p + 1 - j) for j in range(p + 1)) / (p + 1)
for p in (5, 10):
    direct = sum(j**p for j in range(1000))
    print(f'p = {p:2d}: sum_(k<1000) k^p = {direct}  Faulhaber agrees: {faulhaber(p, 1000) == direct}')
print('Bernoulli\'s sum 1^10 + ... + 1000^10 =', sum(j**10 for j in range(1, 1001)))

Bernoulli's number is $91\,409\,924\,241\,424\,243\,424\,241\,924\,242\,500$, which he printed in *Ars Conjectandi*. Its strange regularity is Faulhaber's formula at $n=1000=10^3$: the terms $\frac{n^{11}}{11}+\frac{n^{10}}2+\frac56n^9-n^7+n^5-\frac12n^3+\frac5{66}n$ are powers of $10$ times simple fractions, so their decimal expansions (the repeating $0.0909\ldots$ of $\frac1{11}$, the $0.8333\ldots$ of $\frac56$) overlap in blocks. Note the sum with $k<1000$ differs from Bernoulli's (which includes $k=1000$) by $1000^{10}=10^{30}$.

### P1.3

In [ ]:
import itertools, time
def count_triple(N):
    c = 0
    for i in range(N):
        for j in range(i):
            for l in range(j):
                c += 1
    return c
def count_insertion_worst(N):      # comparisons of insertion sort on a reversed list
    a, c = list(range(N, 0, -1)), 0
    for i in range(1, N):
        x, j = a[i], i - 1
        while j >= 0:
            c += 1
            if a[j] > x:
                a[j + 1] = a[j]; j -= 1
            else:
                break
        a[j + 1] = x
    return c
def newton(values):
    """Newton coefficients Delta^k f(0) from f(0), f(1), ..."""
    coeffs, row = [], list(values)
    while row:
        coeffs.append(row[0]); row = [row[i + 1] - row[i] for i in range(len(row) - 1)]
    return coeffs
x = sp.symbols('x')
for name, f in [('triple loop', count_triple), ('insertion sort, reversed input', count_insertion_worst)]:
    vals = [f(N) for N in range(8)]
    c = newton(vals)
    poly = sp.expand(sum(ci * sp.binomial(x, i) for i, ci in enumerate(c)))
    print(f'{name:32s} counts {vals}  Newton coefficients {c}  ->  {sp.factor(poly)};  check at N=100: {f(100)} = {poly.subs(x, 100)}')

The triple loop costs exactly $\binom n3$ and insertion sort on a reversed list exactly $\binom n2$ comparisons: Newton's coefficients are a single $1$, because the loops count subsets, and $\binom xk$ is the basis of polynomials adapted to $\Delta$. Eight measurements suffice because the cost is a polynomial of degree at most $7$; the prediction at $n=100$ confirms it. For costs that are not polynomials, such as $n\log n$ for merge sort, the differences never vanish and the fitted polynomial is meaningless outside the measured range: Chapter 2 solves such recurrences properly.

### P1.4

In [ ]:
def difference_engine(initial_column, steps):
    """Babbage: hold f(0), Delta f(0), ..., Delta^d f(0) and produce f(0), f(1), ... by additions only."""
    reg = list(initial_column); out = []
    for _ in range(steps):
        out.append(reg[0])
        for i in range(len(reg) - 1):            # each register adds the one below it
            reg[i] += reg[i + 1]
    return out
euler = lambda t: t * t + t + 41
col = newton([euler(t) for t in range(3)])
table = difference_engine(col, 50)
assert table == [euler(t) for t in range(50)]
print('initial column', col, ' first values', table[:8])
print('number of primes among f(0..39):', sum(sp.isprime(v) for v in table[:40]), ' f(40) =', table[40], '=', sp.factorint(table[40]))

The registers start at $41,2,2$ (the second difference of a quadratic is constant), and from then on the machine only adds. All forty values $f(0),\dots,f(39)$ are prime, Euler's famous observation, and $f(40)=41^2$ is not. Babbage's Difference Engine No.~2 held seven orders of differences of $31$-digit numbers; it was finally built from his drawings by the Science Museum in London in 1991, and it works.

### P1.5

In [ ]:
import numpy as np
from math import factorial, comb
def stirling2(N):
    S = [[0] * (N + 1) for _ in range(N + 1)]; S[0][0] = 1
    for m in range(N):
        for j in range(1, m + 2):
            S[m + 1][j] = j * S[m][j] + S[m][j - 1]
    return S
def stirling1(N):
    c = [[0] * (N + 1) for _ in range(N + 1)]; c[0][0] = 1
    for m in range(N):
        for j in range(1, m + 2):
            c[m + 1][j] = m * c[m][j] + c[m][j - 1]
    return c
S2, S1 = stirling2(20), stirling1(20)
print('S2 row 6:', S2[6][1:7], '  S1 row 6:', S1[6][1:7])
assert all(S2[m][j] == sum((-1)**(j - i) * comb(j, i) * i**m for i in range(j + 1)) // factorial(j) for m in range(15) for j in range(m + 1))
rng = np.random.default_rng(1)
def cycles(perm):
    seen, c = np.zeros(len(perm), bool), 0
    for s in range(len(perm)):
        if not seen[s]:
            c += 1; t = s
            while not seen[t]:
                seen[t] = True; t = perm[t]
    return c
N, trials = 10, 200_000
counts = np.bincount([cycles(rng.permutation(N)) for _ in range(trials)], minlength=N + 1)
print(' k   simulated   [10 k]/10!')
for j in range(1, 7):
    print(f'{j:2d}   {counts[j] / trials:.5f}     {S1[N][j] / factorial(N):.5f}')

The rows match Table 1.2 and the explicit formula holds. The simulated frequencies of the number of cycles agree with $\stirlingi{10}k/10!$ to within about $0.002$, the size of the Monte Carlo error $\sqrt{p(1-p)/200\,000}\approx0.001$. Its mean is $H_{10}\approx2.93$, as Tome 3, Chapter 2, showed.

### P1.6

In [ ]:
B = bernoulli_minus(200)
ok = all((B[m] + sum(sp.Rational(1, p) for p in sp.primerange(2, m + 2) if m % (p - 1) == 0)).q == 1 for m in range(2, 201, 2))
print('von Staudt-Clausen holds for every even m <= 200:', ok)
irregular = [p for p in sp.primerange(5, 200) if any(sp.fraction(B[j])[0] % p == 0 for j in range(2, p - 2, 2))]
print('irregular primes below 200:', irregular)
print('B_12 numerator:', sp.fraction(B[12])[0], '  B_32 numerator:', sp.factorint(sp.fraction(B[32])[0]))

Von Staudt–Clausen holds for every even $m\le200$. The irregular primes below $200$ are $37,59,67,101,103,131,149,157$; for instance $37$ divides the numerator of $B_{32}$. There are infinitely many irregular primes (Jensen, 1915), and heuristically about $61\%$ of primes are regular, but nobody has proved that there are infinitely many regular ones. Fermat's last theorem itself was proved by Wiles in 1995, by entirely different methods.